# Hybrid Isolation Forest + Autoencoder Safety Pipeline

This notebook adds a reconstruction-based Autoencoder anomaly detector beside IsolationForest. Both detectors are trained only on non-hazard recordings (`SAFE` and `EMPTY`); ExtraTrees remains the supervised 5-class model. `EMPTY` means no person/object and is non-hazard.

The validation set selects the IsolationForest threshold, Autoencoder reconstruction-error threshold, confidence threshold, distance thresholds, and fusion rule. The test set is evaluated only after selection.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score)

RANDOM_STATE = 42
LABELS = ['SAFE', 'APPROACHING', 'DANGER', 'RETREATING', 'EMPTY']
HAZARD_LABELS = {'APPROACHING', 'DANGER', 'RETREATING'}
NORMAL_LABELS = {'SAFE', 'EMPTY'}
FEATURE_VERSION = 'robust-v3-if-autoencoder'

def find_data_dir():
    candidates = [Path('/kaggle/input/data_blockchan/data/raw'), Path('/kaggle/input/data-blockchan/data/raw'),
                  Path.cwd() / 'ai_model' / 'data' / 'raw', Path.cwd() / 'data' / 'raw']
    for root in candidates:
        if root.is_dir() and list(root.glob('*.csv')):
            return root
    for root in [Path('/kaggle/input'), Path.cwd(), Path.cwd().parent]:
        if root.is_dir():
            matches = [p.parent for p in root.rglob('*.csv') if p.parent.name.lower() == 'raw' and p.parent.parent.name.lower() == 'data']
            if matches:
                return max(set(matches), key=lambda p: len(list(p.glob('*.csv'))))
    raise FileNotFoundError('Could not find data/raw CSV directory')

DATA_DIR = find_data_dir()
print('DATA_DIR:', DATA_DIR)

In [ ]:
RAW_LABELS = set(LABELS)
FEATURE_COLUMNS = ['left_cm', 'right_cm', 'min_distance_cm', 'distance_delta_cm', 'sensor_spread_cm',
                   'distance_median_5', 'distance_std_5', 'approach_speed_cm_s', 'approach_rate_cm_s', 'bad_dt']

def add_features(frame):
    required = {'timestamp', 'elapsed_s', 'left_cm', 'right_cm', 'label', 'source_file'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    out = frame.copy()
    for col in ['left_cm', 'right_cm', 'elapsed_s']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['label'] = out['label'].astype(str).str.upper().str.strip()
    unknown = sorted(set(out['label'].dropna()) - RAW_LABELS)
    if unknown:
        raise ValueError(f'Unknown labels: {unknown}')
    before = len(out)
    out = out.sort_values(['source_file', 'elapsed_s'], kind='stable')
    keep = out['elapsed_s'].notna() & out[['left_cm', 'right_cm']].notna().all(axis=1)
    out = out.loc[keep].copy().reset_index(drop=True)
    out.attrs['dropped_missing_sensor'] = int(before - len(out))
    out['min_distance_cm'] = out[['left_cm', 'right_cm']].min(axis=1)
    out['distance_delta_cm'] = out['left_cm'] - out['right_cm']
    out['sensor_spread_cm'] = out['distance_delta_cm'].abs()
    grouped = out.groupby('source_file', sort=False)
    out['dt_s'] = grouped['elapsed_s'].diff()
    valid_dt = out['dt_s'].between(0.03, 1.0, inclusive='both')
    out['bad_dt'] = (~valid_dt).astype(int)
    out['distance_median_5'] = grouped['min_distance_cm'].transform(lambda s: s.rolling(5, min_periods=1).median())
    out['distance_std_5'] = grouped['min_distance_cm'].transform(lambda s: s.rolling(5, min_periods=2).std()).fillna(0.0)
    smooth_delta = grouped['distance_median_5'].diff()
    out['approach_speed_cm_s'] = smooth_delta.div(out['dt_s'].where(valid_dt)).replace([np.inf, -np.inf], np.nan).clip(-500, 500).fillna(0.0)
    out['approach_rate_cm_s'] = -out['approach_speed_cm_s']
    return out

frames = []
for path in sorted(DATA_DIR.glob('*.csv')):
    frame = pd.read_csv(path)
    frame['source_file'] = path.name
    frames.append(frame)
data = add_features(pd.concat(frames, ignore_index=True))
print('Rows:', len(data), 'Dropped:', data.attrs['dropped_missing_sensor'])
print(data['label'].value_counts().reindex(LABELS, fill_value=0))

In [ ]:
def group_split(frame, test_size, seed, required_test=(), required_train=()):
    y = frame['label'].reset_index(drop=True)
    groups = frame['source_file'].reset_index(drop=True)
    overall = y.value_counts(normalize=True)
    best, score_best = None, float('inf')
    for tr, te in GroupShuffleSplit(n_splits=500, test_size=test_size, random_state=seed).split(frame, y, groups):
        missing = len(set(required_train) - set(y.iloc[tr])) + len(set(required_test) - set(y.iloc[te]))
        balance = sum(abs((y.iloc[te] == c).mean() - overall.get(c, 0.0)) for c in required_test)
        score = 100 * missing + abs(len(te) / len(frame) - test_size) + balance
        if score < score_best:
            score_best, best = score, (tr, te)
    return best

required_test = {'SAFE', 'APPROACHING', 'DANGER', 'RETREATING'}
tr_pool, te = group_split(data, 0.20, RANDOM_STATE, required_test, set(LABELS))
train_pool, test = data.iloc[tr_pool].reset_index(drop=True), data.iloc[te].reset_index(drop=True)
tr, va = group_split(train_pool, 0.25, RANDOM_STATE + 1, required_test, set(LABELS))
train, validation = train_pool.iloc[tr].reset_index(drop=True), train_pool.iloc[va].reset_index(drop=True)

X_train, y_train = train[FEATURE_COLUMNS], train['label']
X_val, y_val = validation[FEATURE_COLUMNS], validation['label']
X_test, y_test = test[FEATURE_COLUMNS], test['label']
normal_train = X_train[y_train.isin(NORMAL_LABELS)]
classifier = ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
classifier.fit(X_train, y_train)

if_model = Pipeline([('scale', StandardScaler()), ('detector', IsolationForest(n_estimators=500, contamination=0.05, random_state=RANDOM_STATE, max_features=1.0))])
if_model.fit(normal_train)

# MLPRegressor is used as a tabular autoencoder. Both input and target
# must be in the same standardized space; otherwise the reconstruction
# error is meaningless and can explode to 1e8-scale values.
ae_scaler = StandardScaler().fit(normal_train)
normal_train_scaled = ae_scaler.transform(normal_train)
autoencoder = MLPRegressor(hidden_layer_sizes=(32, 12, 32), activation='relu', solver='adam', learning_rate_init=1e-3, max_iter=600, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_STATE)
autoencoder.fit(normal_train_scaled, normal_train_scaled)
print('Train/val/test:', len(train), len(validation), len(test))
print('Normal rows for IF+AE:', len(normal_train))

## IF + Autoencoder scores

IsolationForest returns a higher anomaly score for more abnormal points. The Autoencoder score is mean squared reconstruction error after the shared scaler. Thresholds are selected from validation only.

In [ ]:
def if_anomaly_score(model, X):
    return -model.decision_function(X)

def ae_reconstruction_error(model, scaler, X):
    scaled = scaler.transform(X)
    reconstructed = model.predict(scaled)
    return np.mean((scaled - reconstructed) ** 2, axis=1)

if_train_score = if_anomaly_score(if_model, normal_train)
ae_train_score = ae_reconstruction_error(autoencoder, ae_scaler, normal_train)
if_val_score = if_anomaly_score(if_model, X_val)
ae_val_score = ae_reconstruction_error(autoencoder, ae_scaler, X_val)
print('IF normal score quantiles:', np.quantile(if_train_score, [0.90, 0.95, 0.99]))
print('AE normal error quantiles:', np.quantile(ae_train_score, [0.90, 0.95, 0.99]))

In [ ]:
def hybrid_states(X, classifier, if_model, autoencoder, ae_scaler, warning_cm, danger_cm, confidence_threshold, if_threshold, ae_threshold, fusion='and'):
    pred = classifier.predict(X)
    confidence = classifier.predict_proba(X).max(axis=1)
    if_flag = if_anomaly_score(if_model, X) >= if_threshold
    ae_flag = ae_reconstruction_error(autoencoder, ae_scaler, X) >= ae_threshold
    minimum = X['min_distance_cm'].to_numpy()
    states, reasons = [], []
    for i in range(len(X)):
        if pred[i] == 'EMPTY':
            states.append('EMPTY'); reasons.append('NO_OBJECT'); continue
        if minimum[i] <= danger_cm:
            states.append('EMERGENCY'); reasons.append('HARD_DANGER_THRESHOLD'); continue
        if minimum[i] <= warning_cm:
            states.append('WARNING'); reasons.append('HARD_WARNING_THRESHOLD'); continue
        classifier_vote = pred[i] in HAZARD_LABELS and confidence[i] >= confidence_threshold
        anomaly_vote = (if_flag[i] and ae_flag[i]) if fusion == 'and' else (if_flag[i] or ae_flag[i])
        active = classifier_vote and anomaly_vote
        states.append('WARNING' if active else 'SAFE')
        reasons.append('AI_IF_AE_' + fusion.upper() if active else 'NORMAL')
    return pd.DataFrame({'state': states, 'reason': reasons, 'classifier_state': pred, 'classifier_confidence': confidence, 'if_anomaly': if_flag.astype(int), 'ae_anomaly': ae_flag.astype(int)}, index=X.index)

def safety_metrics(states, target):
    actual = target.isin(HAZARD_LABELS).to_numpy().astype(int)
    predicted = states.isin(['WARNING', 'EMERGENCY']).to_numpy().astype(int)
    non_hazard = actual == 0
    cm = confusion_matrix(actual, predicted, labels=[0, 1])
    return {'accuracy': float(accuracy_score(actual, predicted)), 'balanced_accuracy': float(balanced_accuracy_score(actual, predicted)), 'precision': float(precision_score(actual, predicted, zero_division=0)), 'recall': float(recall_score(actual, predicted, zero_division=0)), 'f1': float(f1_score(actual, predicted, zero_division=0)), 'non_hazard_fpr': float(predicted[non_hazard].mean()) if non_hazard.any() else 0.0, 'empty_recall': float(((states == 'EMPTY') & (target == 'EMPTY')).sum() / max((target == 'EMPTY').sum(), 1)), 'confusion_matrix': cm.tolist()}

if_grid = np.quantile(if_train_score, [0.90, 0.95, 0.97, 0.99])
ae_grid = np.quantile(ae_train_score, [0.90, 0.95, 0.97, 0.99])
rows = []
for if_threshold in if_grid:
    for ae_threshold in ae_grid:
        for fusion in ['and', 'or']:
            for confidence in [0.55, 0.70, 0.85]:
                states = hybrid_states(X_val, classifier, if_model, autoencoder, ae_scaler, 45.0, 30.0, confidence, if_threshold, ae_threshold, fusion)['state']
                metrics = safety_metrics(states, y_val)
                metrics.update({'if_threshold': float(if_threshold), 'ae_threshold': float(ae_threshold), 'fusion': fusion, 'confidence': confidence})
                metrics['utility'] = 2 * metrics['recall'] + metrics['f1'] + metrics['balanced_accuracy'] - 2 * metrics['non_hazard_fpr']
                rows.append(metrics)
tuning = pd.DataFrame(rows).sort_values(['utility', 'recall', 'f1'], ascending=False).reset_index(drop=True)
best = tuning.iloc[0].to_dict()
print(tuning.head(10).to_string(index=False))
print('Selected IF+AE validation configuration:', best)

In [ ]:
test_hybrid = hybrid_states(X_test, classifier, if_model, autoencoder, ae_scaler, 45.0, 30.0, float(best['confidence']), float(best['if_threshold']), float(best['ae_threshold']), str(best['fusion']))
test_metrics = safety_metrics(test_hybrid['state'], y_test)
print('Final IF + Autoencoder test metrics:')
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
print(test_hybrid['state'].value_counts())
actual = y_test.isin(HAZARD_LABELS).astype(int).to_numpy()
predicted = test_hybrid['state'].isin(['WARNING', 'EMERGENCY']).astype(int).to_numpy()
ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(actual, predicted, labels=[0, 1]), display_labels=['NON_HAZARD', 'HAZARD']).plot(cmap='Blues', values_format='d')
plt.title('IsolationForest + Autoencoder hybrid test')
plt.tight_layout()
plt.show()

In [ ]:
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'artifacts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifact = {'classifier': classifier, 'isolation_forest': if_model, 'autoencoder': autoencoder, 'autoencoder_scaler': ae_scaler, 'feature_columns': FEATURE_COLUMNS, 'label_names': LABELS, 'hazard_labels': sorted(HAZARD_LABELS), 'normal_labels': sorted(NORMAL_LABELS), 'thresholds_cm': {'warning': 45.0, 'danger': 30.0}, 'ai_confidence_threshold': float(best['confidence']), 'if_anomaly_threshold': float(best['if_threshold']), 'ae_reconstruction_threshold': float(best['ae_threshold']), 'anomaly_fusion': str(best['fusion']), 'feature_version': FEATURE_VERSION, 'target_policy': 'HAZARD=APPROACHING|DANGER|RETREATING; EMPTY=NO_OBJECT'}
joblib.dump(artifact, OUTPUT_DIR / 'hybrid_model_if_autoencoder.joblib')
payload = {'validation_selected_config': best, 'test_metrics': test_metrics, 'test_class_support': y_test.value_counts().reindex(LABELS, fill_value=0).to_dict()}
(OUTPUT_DIR / 'hybrid_metrics_if_autoencoder.json').write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'hybrid_model_if_autoencoder.joblib')
print('Saved:', OUTPUT_DIR / 'hybrid_metrics_if_autoencoder.json')

## Deployment note

At inference time, apply exactly the same feature engineering and StandardScaler pipelines. `if_anomaly` and `ae_anomaly` are complementary: IsolationForest captures isolation/density outliers, while the Autoencoder captures samples that cannot be reconstructed from the learned non-hazard manifold. The default `and` fusion is conservative; validate `or` separately if missing hazards is more costly than false alarms.